# Single-session optic-flow analysis

A linear, per-session walk-through of the eye/face optic-flow pipeline: **load a frame → define the
ROIs → remove the corneal reflection → run per-ROI flow tracking + pupil tracking → inspect**. Each
step drives the reusable modules in `common/`, so this notebook is the *front end*,
not a reimplementation.

**Pipeline order** (what reads what):

| step | module | writes |
|---|---|---|
| 0  load a frame | — | — |
| 1  define ROIs | you, on the frame | `roi_config_<mouse>.json` |
| 2  remove the IR glint | `common/remove_reflection.py` | `<mouse>_..._noreflection.mp4` |
| 3  build the session contract | `common/session_config.py` | `session.json` |
| 4  per-ROI optic flow | `common/compute_roi_flow.py` | `opticflow/opticflow_<roi>_<metric>.npy` |
| 5  pupil tracking | `common/detectors/segment_pupil.py` | `opticflow/pupil_track.npz` |

Two videos are kept and stay frame-aligned: the **original** (glint intact — the pupil fit anchors on
the glint) and the **noreflection** copy (glint inpainted — the flow reads this). The session folder
also needs the game **`log.json`** for `build_session`.

> **`RUN_FULL`** (set in the config cell) guards every step that *writes into the session* or decodes
> the whole video. Left `False`, a "Run All" only does the cheap previews and overwrites nothing — flip
> it to `True` to run the real, full-session pipeline.

In [ ]:
# ── imports + make ../common importable ───────────────────────────────────────
import sys, json, time
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

_HERE = Path.cwd()
# this notebook lives in common/ (next to the modules); fall back to ../common if run from elsewhere
_COMMON = _HERE if (_HERE / 'compute_roi_flow.py').exists() else (_HERE.parent / 'common')
assert (_COMMON / 'compute_roi_flow.py').exists(), f'cannot locate common/ from {_HERE}'
sys.path.insert(0, str(_COMMON))
sys.path.insert(0, str(_COMMON / 'detectors'))

import remove_reflection as rr
import compute_roi_flow as rflow
import segment_pupil as sp
import session_config as scfg
import fps as fpsmod
print('modules loaded from', _COMMON)

In [ ]:
# ── CONFIG: point at ONE session folder ───────────────────────────────────────
# Sessions live as  <MAIN_DIR> / <MOUSE_ID> / <date> .  The leaf is the recording folder, e.g.
# 'JPAS_168_2026-07-22_10;46;59'. It holds the ORIGINAL eye video (glint intact) + log.json.
MAIN_DIR = '/path/to/MAIN_DIR'    # <-- server root that holds the animal folders
MOUSE_ID = 'MOUSE_ID'             # <-- animal folder name (also names the outputs)
date     = ''                     # <-- session sub-folder (recording date/time); '' = none

SESSION_DIR = Path(MAIN_DIR) / MOUSE_ID / date       # empty parts are dropped by pathlib

# The safety switch. False = cheap previews only, nothing written. True = the real full-session run.
RUN_FULL = False

# window used by the PREVIEW cells (cheap: a few hundred frames)
PREVIEW_LO, PREVIEW_HI = 5000, 5200
PREVIEW_FRAME = 5000            # single frame for the ROI / reflection previews

SESSION_DIR = SESSION_DIR.resolve()
assert SESSION_DIR.exists(), f'session folder does not exist: {SESSION_DIR}  (set MAIN_DIR / MOUSE_ID / date)'

def _resolve_videos(session_dir):
    '''(ORIGINAL, NOREFLECTION). Prefer the authoritative paths in session.json; else discover on disk,
    EXCLUDING the videos this pipeline itself generates (so a flow-overlay/annotated clip can't win).'''
    d = Path(session_dir); orig = noref = None
    sp = d / 'session.json'
    if sp.exists():
        s = json.load(open(sp))
        for key, var in (('video_original', 'orig'), ('video_noreflection', 'noref')):
            v = s.get(key)
            if v:
                p = Path(v) if Path(v).is_absolute() else d / v
                if p.exists():
                    if var == 'orig': orig = p
                    else: noref = p
    if noref is None:
        nr = sorted(d.glob('*noreflection*.mp4')); noref = nr[0] if nr else None
    if orig is None:
        gen = ('noreflection', 'ui', 'viz', 'opticflow', 'flow_overlay', 'flow', 'overlay',
               'annotated', 'clip', 'reconstruct', 'mouse_view', 'still', 'saccade', 'lick', 'groom')
        cands = [p for p in sorted(d.glob('*.mp4')) if not any(t in p.name.lower() for t in gen)]
        orig = cands[0] if cands else None
    return orig, noref

ORIGINAL_VIDEO, NOREF_VIDEO = _resolve_videos(SESSION_DIR)
print('main dir  :', MAIN_DIR)
print('session   :', SESSION_DIR)
print('mouse     :', MOUSE_ID, '| date:', date or '(none)')
print('original  :', ORIGINAL_VIDEO.name if ORIGINAL_VIDEO else None)
print('noreflect :', NOREF_VIDEO.name if NOREF_VIDEO else '(not generated yet)')
print('RUN_FULL  :', RUN_FULL, '(False = previews only, writes nothing)')

## 0 — Load a frame

Grab one representative frame from the original video and look at it. Frame indices are the same in the
original and the (later) noreflection copy, so any index you pick here is valid everywhere.

In [ ]:
assert ORIGINAL_VIDEO is not None, (
    f'no original (with-reflection) .mp4 in {SESSION_DIR}\n'
    '   -> the folder needs the eye video whose name does NOT contain noreflection / ui / viz')
cap = cv2.VideoCapture(str(ORIGINAL_VIDEO))
assert cap.isOpened(), f'cannot OPEN {ORIGINAL_VIDEO} (missing codec or unreadable file)'
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
NFRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); VFPS = cap.get(cv2.CAP_PROP_FPS) or 30.0

# a short session may have fewer frames than PREVIEW_FRAME -> clamp it into range
if 0 < NFRAMES <= PREVIEW_FRAME:
    PREVIEW_FRAME = max(0, NFRAMES // 2)
    print(f'PREVIEW_FRAME was past the end -> using frame {PREVIEW_FRAME} of {NFRAMES}')

cap.set(cv2.CAP_PROP_POS_FRAMES, PREVIEW_FRAME)
ok, frame = cap.read()
if not ok:                                     # some codecs can't seek -> read sequentially from 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    for _ in range(PREVIEW_FRAME + 1):
        ok, frame = cap.read()
        if not ok:
            break
cap.release()
assert ok, (f'could not read frame {PREVIEW_FRAME} of {ORIGINAL_VIDEO.name} '
            f'(video reports {NFRAMES} frames). Lower PREVIEW_FRAME in the config cell.')
print(f'{ORIGINAL_VIDEO.name}\n{W}x{H}px  {VFPS:.2f} fps  {NFRAMES} frames  '
      f'({NFRAMES/VFPS/60:.1f} min)  | preview frame {PREVIEW_FRAME}')

plt.figure(figsize=(11, 6.2))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f'{MOUSE_ID}  frame {PREVIEW_FRAME}'); plt.axis('off'); plt.show()

## 1 — Define the ROIs (stepwise)

An ROI is a box `[x1, y1, x2, y2]` in **original-video pixels**. We do this in small steps so you never
have to scroll: **1a** shows the frame under a numbered grid to read coordinates off; **1b** does the
**load-or-define** (use this session's saved `roi_config` if there is one, else start empty) and loads
the helpers; **1c** adds any **missing** ROIs and draws them **once**; **1d** zooms a single ROI to
fine-tune its edges; **1e** saves. The eye boxes (`left_eye`, `left_fovea`) matter twice — their glint
is what step 2 inpaints, and `left_fovea` gates the pupil fit in step 5.

> **Already have a `roi_config`?** 1b loads it and tells you so — you can skip straight to **step 2**.
> Run 1c only to add a box you don't have yet, or `set_roi(...)` to edit one.

**1a — the coordinate grid.** Major gridlines + tick labels every `GRID_STEP` px, faint minor lines at
the half-step. Read the corners of the box you want off the axes, then type them into `set_roi` below.

In [ ]:
GRID_STEP = 100   # px between labelled gridlines -- lower it (e.g. 50) to read finer

def show_grid(rois=None, step=GRID_STEP, region=None, title=None, figsize=(13, 7.6)):
    '''Show the frame with a pixel ruler + grid, and (optionally) the ROIs drawn on top.
       region=(x1,y1,x2,y2) zooms in; rois=dict draws boxes+labels.'''
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for name, r in (rois or {}).items():
        x1, y1, x2, y2 = r['bbox']; col = [c / 255 for c in r['color_bgr'][::-1]]  # BGR->RGB
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=col, lw=2))
        ax.text(x1 + 2, y1 - 4, name, color=col, fontsize=9, fontweight='bold')
    ax.set_xticks(np.arange(0, W + 1, step)); ax.set_yticks(np.arange(0, H + 1, step))
    ax.set_xticks(np.arange(0, W + 1, max(10, step // 2)), minor=True)
    ax.set_yticks(np.arange(0, H + 1, max(10, step // 2)), minor=True)
    ax.grid(which='major', color='#ffd000', alpha=0.55, lw=0.8)
    ax.grid(which='minor', color='#ffd000', alpha=0.20, lw=0.5)
    ax.tick_params(labelsize=8)
    if region:
        rx1, ry1, rx2, ry2 = region; ax.set_xlim(rx1, rx2); ax.set_ylim(ry2, ry1)  # y inverted
    ax.set_title(title or f'{MOUSE_ID}  frame {PREVIEW_FRAME} — read x,y off the grid (px)')
    plt.tight_layout(); plt.show()

show_grid()

**1b — load-or-define + helpers.** `START_FROM_EXISTING` decides whether to begin from this session's
saved `roi_config` (loaded here → skip to step 2) or from an empty set. This cell also defines
`set_roi(name, x1, y1, x2, y2)` (add/replace one box — **silent**, so a batch of calls makes no figures)
and `drop_roi(name)` (remove one). It sets `HAVE_ROI` so 1c can tell whether a config was already loaded.

In [ ]:
START_FROM_EXISTING = True      # False -> ignore any saved config and define the ROIs from scratch

_cfg = sorted(SESSION_DIR.glob('roi_config*.json'))
HAVE_ROI = bool(_cfg)
if START_FROM_EXISTING and HAVE_ROI:
    ROIS = {k: {'bbox': list(v['bbox']), 'color_bgr': list(v.get('color_bgr', [0, 255, 0]))}
            for k, v in json.load(open(_cfg[0])).items()}
    print('roi_config FOUND ->', _cfg[0].name, '->', list(ROIS))
    print('ROIs already defined -- you can SKIP 1c/1d and jump to step 2.')
    print('Run 1c only to ADD a missing ROI or EDIT one (set_roi always overwrites).')
else:
    ROIS = {}
    print(('roi_config exists but START_FROM_EXISTING=False' if HAVE_ROI else 'no roi_config found')
          + ' -> DEFINE the ROIs from scratch in 1c below')

_PALETTE = [(0,255,0), (0,0,255), (255,0,255), (0,128,255), (255,165,0), (0,255,255), (255,255,0)]

def set_roi(name, x1, y1, x2, y2, color=None, draw=False):
    '''Add/replace ONE ROI. SILENT by default (so a whole batch of calls makes no figures);
       pass draw=True to redraw the grid right here after this one box.'''
    x1, x2 = sorted((int(x1), int(x2))); y1, y2 = sorted((int(y1), int(y2)))
    color = color or ROIS.get(name, {}).get('color_bgr') or _PALETTE[len(ROIS) % len(_PALETTE)]
    ROIS[name] = {'bbox': [x1, y1, x2, y2], 'color_bgr': list(color)}
    if draw:
        show_grid(rois=ROIS, title=f'set {name} = [{x1}, {y1}, {x2}, {y2}]   ({len(ROIS)} ROIs total)')

def drop_roi(name, draw=True):
    ROIS.pop(name, None)
    if draw:
        show_grid(rois=ROIS, title=f'dropped {name}   ({len(ROIS)} ROIs total)')

print('helpers ready:  set_roi(name, x1,y1,x2,y2) · drop_roi(name)')

**1c — define / top up the ROIs (one frame).** If 1b loaded a saved `roi_config`, the `seed(...)` lines
add only the ROIs that are **missing** — a saved config is never overwritten; starting from empty they
lay down the template. Edit the numbers (read corners off the 1a grid) and re-run — the frame under the
cell redraws **once** with every ROI. To change a box that already exists, call
`set_roi(name, x1, y1, x2, y2)` directly (it overwrites). This session uses `whisker_left/right`, `nose`,
`mouth`, `paw`, `left_eye`, `left_fovea`.

In [ ]:
# LOAD-OR-DEFINE: if 1b loaded a saved roi_config, these lines add ONLY the ROIs that are MISSING --
# a saved config is never overwritten. Starting from empty, they seed the template. To EDIT a box that
# already exists, call set_roi(name, x1,y1,x2,y2) yourself (it always overwrites). Read corners off 1a.
def seed(name, *box):
    if name not in ROIS:
        set_roi(name, *box)          # silent; the single show_grid at the end draws them all once

seed('left_eye',   615, 168, 710, 233)
seed('left_fovea', 630, 175, 676, 206)
# seed('whisker_right', 875,  20, 1200, 220)
# seed('whisker_left',  425, 285,  750, 450)
# seed('nose',          840, 245,  925, 330)
# seed('mouth',         760, 333,  910, 420)
# seed('paw',           360, 500,  740, 680)

show_grid(rois=ROIS, title=f'{len(ROIS)} ROIs: ' + ', '.join(ROIS))   # ONE frame with every ROI drawn

**1d — zoom to fine-tune one box.** Pass a ROI name to see it enlarged on a fine grid, so you can read
the exact edges and correct them with another `set_roi` in 1c.

In [ ]:
def zoom_roi(name, pad=40, step=20):
    x1, y1, x2, y2 = ROIS[name]['bbox']
    show_grid(rois={name: ROIS[name]}, step=step, figsize=(9, 7),
              region=(max(0, x1 - pad), max(0, y1 - pad), min(W, x2 + pad), min(H, y2 + pad)),
              title=f'{name} = [{x1},{y1},{x2},{y2}]  (grid {step}px) — fine-tune the edges')

zoom_roi('left_eye')      # <-- change the name to inspect any ROI

**1e — save.** Writes `roi_config_<mouse>.json` (guarded by `RUN_FULL`), which every downstream step
reads.

*Prefer to drag boxes instead of typing coordinates? If your Jupyter has `ipympl`, run `%matplotlib
widget` in a cell, then use `matplotlib.widgets.RectangleSelector` (its `onselect` gives you the
`x1,y1,x2,y2` to paste into `set_roi`). On a local display, `cv2.selectROI("roi", frame)` returns the
same four numbers. Both just feed `set_roi` — the grid workflow above needs neither.*

In [ ]:
# SAVE the ROI config (guarded). Downstream (build_session, remove_reflection) reads this file.
roi_path = SESSION_DIR / f'roi_config_{MOUSE_ID}.json'
if RUN_FULL:
    json.dump({n: {'bbox': [int(v) for v in r['bbox']],
                   'color_bgr': [int(c) for c in r['color_bgr']]} for n, r in ROIS.items()},
              open(roi_path, 'w'), indent=2)
    print('wrote', roi_path)
else:
    print('RUN_FULL is False -> not writing', roi_path.name, '(flip RUN_FULL to save)')

## 2 — Remove the corneal reflection (IR glint)

The IR LEDs leave a near-white specular glint on the cornea. It sits over the pupil and confuses both
the flow and the pupil fit, so it is inpainted out of the **eye boxes only** — producing the
`*_noreflection.mp4` that the flow step reads. First preview the mask + inpaint on one frame (adjust
`threshold` if the mask misses the glint or eats too much), then write the full video.

In [ ]:
# preview the glint mask + inpaint on the eye crop (cheap, one frame). PANEL 1 shows WHERE the eye box
# sits on the whole frame -- if the bright glint is not inside the box, the box is wrong for THIS session
# (redefine it in step 1) and there is nothing to remove.
eye_names = [n for n in ('left_eye', 'left_fovea', 'eye', 'fovea') if n in ROIS]
assert eye_names, 'need an eye/fovea ROI for reflection removal -- define one in step 1'
eb = ROIS[eye_names[0]]['bbox']
crop, mask, cleaned = rr.preview(ORIGINAL_VIDEO, eb, frame=PREVIEW_FRAME, threshold=rr.THRESHOLD)
n_glint = int((crop > rr.THRESHOLD).sum())
print(f'{ORIGINAL_VIDEO.name}  frame {PREVIEW_FRAME}  eye box {eye_names[0]}={eb}')
print(f'glint pixels >{rr.THRESHOLD}: {n_glint} | mask px: {(mask>0).sum()} | '
      f'crop max {crop.max()} -> after inpaint {cleaned.max()}')
if n_glint == 0:
    print('WARNING: nothing brighter than the threshold in this box -> either the eye ROI is OFF the '
          'glint (redefine it in step 1), this video is ALREADY de-reflected, or lower rr.THRESHOLD.')

ctx = frame.copy()
x1, y1, x2, y2 = eb; cv2.rectangle(ctx, (x1, y1), (x2, y2), (0, 255, 0), 2)
z = 6; sz = (crop.shape[1] * z, crop.shape[0] * z)
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(cv2.cvtColor(ctx, cv2.COLOR_BGR2RGB)); ax[0].axis('off')
ax[0].set_title(f'{eye_names[0]} box on frame {PREVIEW_FRAME}')
for a, im, t in zip(ax[1:], [crop, mask, cleaned], ['original eye crop', 'glint mask', 'inpainted']):
    a.imshow(cv2.resize(im, sz, interpolation=cv2.INTER_NEAREST), cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# WRITE the FULL noreflection video -- the WHOLE session, every frame. It is GENERATED from the ORIGINAL
# (glint-intact) video by inpainting the eye boxes. Skipped when one ALREADY EXISTS (any name, from
# session.json or a *noreflection*.mp4 on disk) so a re-run doesn't redo the minutes-long decode.
eye_bboxes = [ROIS[n]['bbox'] for n in eye_names]
if NOREF_VIDEO and NOREF_VIDEO.exists():
    print('noreflection video ALREADY generated ->', NOREF_VIDEO.name, '   (delete it to regenerate)')
elif RUN_FULL:
    noref_path = SESSION_DIR / f'{MOUSE_ID}_noreflection.mp4'
    rr.generate_noreflection_video(ORIGINAL_VIDEO, noref_path, eye_bboxes,   # hi=None => WHOLE video
                                   threshold=rr.THRESHOLD, dilate_iter=rr.DILATE_ITER,
                                   inpaint_radius=rr.INPAINT_RADIUS)
    NOREF_VIDEO = noref_path
else:
    print('RUN_FULL is False -> NOT generating the noreflection video (nothing written).')
    print(f'   set RUN_FULL = True to inpaint the WHOLE session -> {MOUSE_ID}_noreflection.mp4')

## 3 — Build the session contract (`session.json`)

`build_session` discovers the two videos + the ROI config + `log.json`, derives fps from the log
camera table and cross-checks the video, classifies the task, runs the camera-shift check, and writes
**`session.json`** — the single file every downstream step reads. It **merges** into any existing
`session.json`, so manual keys (e.g. `view_scale`) survive a rebuild.

Like steps 4 and 5, this cell **loads `session.json` if it already exists** (the camera check is slow,
no need to redo it every run) and only builds when it's missing — set `REBUILD_SESSION=True` to force a
re-derive from the log.

In [ ]:
# LOAD session.json if it already exists (fast) -- only BUILD when it's missing or you force a rebuild.
# build_session re-derives fps + runs the camera-shift check, so we don't want to redo it every run.
REBUILD_SESSION = False        # True -> re-derive from the log even if session.json is present
_sp = SESSION_DIR / 'session.json'
if _sp.exists() and not REBUILD_SESSION:
    sess = json.load(open(_sp))
    print('session.json exists -> loaded (set REBUILD_SESSION=True to re-derive from the log)')
elif RUN_FULL or REBUILD_SESSION:
    sess = scfg.build_session(MOUSE_ID, str(SESSION_DIR), write=True, verbose=True)
else:
    sess = scfg.build_session(MOUSE_ID, str(SESSION_DIR), write=False, verbose=True)
    print('(built in-memory; not written because RUN_FULL is False)')
print('\ntask:', sess.get('task_type'), '| fps:', round(sess.get('fps', 0), 3),
      '| frames:', sess.get('n_frames'), '| camera_moved:', sess.get('camera_moved'))
print('rois:', list(sess.get('rois', {})))

## 4 — Per-ROI optic flow

`compute_roi_flow` runs Farneback per ROI (on the 0.5-scaled noreflection frame, padded then sliced)
and averages the 8 flow metrics — `mag, x, y, angle, coherence, variance, divergence, radial` — into
`opticflow/opticflow_<roi>_<metric>.npy` (one value per frame). Preview a short window first (returns
the arrays without writing), then the full-run cell **checks for the finished output
(`opticflow_metadata_<mouse>.json`) and just LOADS it if it already exists** — so re-running the
notebook doesn't redo the ~30 min decode — otherwise it computes the whole session.

In [ ]:
# PREVIEW: a few hundred frames, nothing written -- returns {roi: {metric: array}}
arr = rflow.run(str(SESSION_DIR), lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)

t = np.arange(PREVIEW_LO, PREVIEW_HI)
fig, ax = plt.subplots(figsize=(12, 4))
for roi in [r for r in ('paw', 'whisker_left', 'whisker_right', 'mouth', 'nose') if r in arr]:
    ax.plot(t, arr[roi]['mag'][PREVIEW_LO:PREVIEW_HI], lw=1.1, label=roi)
ax.set_xlabel('frame'); ax.set_ylabel('|flow| (mag)')
ax.set_title(f'per-ROI motion energy, frames {PREVIEW_LO}-{PREVIEW_HI}'); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

In [ ]:
# ── FULL per-ROI flow: LOAD if already computed, else COMPUTE ──────────────────
# opticflow_metadata_<mouse>.json is written when a full run finishes, so it marks "already done".
# If it (and the per-ROI .npy arrays) are present, just load them -- no need to redo the ~30 min decode.
ofd = SESSION_DIR / 'opticflow'
_meta = sorted(ofd.glob('opticflow_metadata_*.json'))
_have = bool(_meta) and all((ofd / f'opticflow_{roi}_mag.npy').exists() for roi in sess['rois'])
if _have:
    print('optic flow already computed ->', _meta[0].name, '(loading, not recomputing)')
    flow = {roi: {m: np.load(ofd / f'opticflow_{roi}_{m}.npy') for m in rflow.METRICS}
            for roi in sess['rois']}
elif RUN_FULL:
    print('no optic flow found -> computing the whole session (~30 min)...')
    flow = rflow.run(str(SESSION_DIR), write=True)       # writes the .npy arrays + the metadata JSON
else:
    flow = None
    print('no optic flow found, RUN_FULL is False -> set RUN_FULL=True to compute it '
          '(the step-4 preview above is unwritten)')

## 5 — Pupil tracking

`segment_pupil` fits the dark iris disc anchored on the corneal glint, on the **original** (glint-intact)
video, gated to `left_fovea`. **QC first**: `overlay_grid` renders a contact sheet of the fit on sample
frames — *look at it* before trusting the numbers (this fit is per-animal and only approximate for
pupil size; position + blink are the reliable parts). The fit cell then **loads `pupil_track.npz` if it
already exists** (no refit) and otherwise fits the session → `pupil_track.npz`.

In [ ]:
# QC contact sheet -- writes a PNG to debug/, always safe to run
qc_frames = list(range(PREVIEW_LO, PREVIEW_LO + 180, 30))
qc_png = SESSION_DIR / 'debug' / 'pupil_overlay_qc.png'
sp.overlay_grid(str(SESSION_DIR), qc_frames, out_path=str(qc_png))
plt.figure(figsize=(13, 5)); plt.imshow(plt.imread(qc_png)); plt.axis('off')
plt.title('pupil fit QC -- eyeball this before trusting radius'); plt.show()

In [ ]:
# ── pupil track: LOAD if already computed, else COMPUTE ────────────────────────
# pupil_track.npz is the finished pupil fit; if it's there just load the radius, don't refit.
_pt = SESSION_DIR / 'opticflow' / 'pupil_track.npz'
if _pt.exists():
    r = np.load(_pt)['radius']
    print('pupil already tracked ->', _pt.name, '(loading, not refitting)')
elif RUN_FULL:
    print('no pupil track found -> fitting the whole session...')
    r = sp.run(str(SESSION_DIR), write=True)['radius']       # writes opticflow/pupil_track.npz
else:
    r = sp.run(str(SESSION_DIR), lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)['radius']
    print('no pupil track, RUN_FULL is False -> fit a short preview window (not written)')
print(f'radius: valid {100*np.mean(~np.isnan(r)):.1f}%  median {np.nanmedian(r):.1f} px')

## 6 — Inspect the outputs

Everything downstream (grooming, licking, whisker, eye events, the trial report) reads these arrays.
A quick look: per-ROI motion energy + the pupil radius over the preview window.

In [ ]:
ofdir = SESSION_DIR / 'opticflow'
def load_metric(roi, metric):
    p = ofdir / f'opticflow_{roi}_{metric}.npy'
    return np.load(p) if p.exists() else None

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for roi in ('paw', 'whisker_left', 'whisker_right', 'mouth'):
    m = load_metric(roi, 'mag')
    src = m if m is not None else (arr[roi]['mag'] if roi in arr else None)
    if src is not None:
        ax[0].plot(np.arange(PREVIEW_LO, PREVIEW_HI), src[PREVIEW_LO:PREVIEW_HI], lw=1.1, label=roi)
ax[0].set_ylabel('|flow|'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.2)
ax[0].set_title(f'{MOUSE_ID}  outputs over frames {PREVIEW_LO}-{PREVIEW_HI}')
ax[1].plot(np.arange(PREVIEW_LO, PREVIEW_HI), r[PREVIEW_LO:PREVIEW_HI], color='#8e44ad', lw=1.2)
ax[1].set_ylabel('pupil radius (px)'); ax[1].set_xlabel('frame'); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()

print('optic-flow .npy files:', len(sorted(ofdir.glob('opticflow_*_*.npy'))) if ofdir.exists() else 0)
print('\\nNext, per-animal detectors run on these arrays (see ../common/detectors/):')
print('  compute_eye_events · detect_grooming · detect_licking · compute_mouth_state · detect_saccades')

## 7 — Find good example frames: `browse()` and `present()`

Two helpers so you can hunt for a slide frame fast instead of hard-coding one:

- **`browse(kind, offset_ms=, avg_ms=)`** — a contact sheet of the top candidate frames for an event,
  ranked by ROI motion, with the frame indices printed. Eyeball it, copy the index you like.
- **`present(frame= | kind=, offset_ms=, avg_ms=)`** — the full presentation figure for ONE frame: the
  frame, each ROI boxed + labelled with its motion, the flow-arrow field, and the per-ROI traces around
  it. Saves to `<session>/debug/roi_motion_frame.png`.

**`kind`** = which event: `reward` / `banish` / `unbanish` (log collections) · `lick` (licking bouts) ·
`groom` · `blink` · `saccade` · `resteer` (joystick) · `motion` (peak |flow|).
**`offset_ms`** shifts relative to the event — negative = before (e.g. `-500` = 500 ms before a reward).
**`avg_ms`** averages each ROI's |flow| over a window centred on the frame (e.g. `1000` = a 1-second
average, steadier than one noisy frame; `0` = the instantaneous value).
Event kinds beyond the log ones need their detector to have run (`lick`→detect_licking,
`groom`→detect_grooming, `blink`→compute_eye_events, `saccade`→detect_saccades).

In [ ]:
# ── shared setup + the event catalogue ────────────────────────────────────────
of = SESSION_DIR / 'opticflow'
mag = {roi: np.load(of / f'opticflow_{roi}_mag.npy')
       for roi in ROIS if (of / f'opticflow_{roi}_mag.npy').exists()}
if mag:
    LO, HI, WHERE = 1, len(next(iter(mag.values()))), 'full session'
elif 'arr' in dir():
    mag = {roi: arr[roi]['mag'] for roi in arr}
    LO, HI, WHERE = PREVIEW_LO + 1, PREVIEW_HI, f'preview {PREVIEW_LO}-{PREVIEW_HI}'
else:
    raise RuntimeError('run step 4 (per-ROI optic flow) first -- no motion arrays yet')
FACIAL = [r for r in mag if r not in ('left_eye', 'left_fovea')]
_LOG = json.load(open(SESSION_DIR / 'log.json')); _FM = fpsmod.frame_to_ms(_LOG); FPS = float(sess['fps'])
_ms2f = lambda t: int(np.argmin(np.abs(_FM - t)))
_span = lambda ms: int(round(ms / 1000 * FPS))

def _npz(name):
    p = of / name
    if not p.exists():
        raise FileNotFoundError(f'{name} not in opticflow/ -- run the detector that writes it')
    return np.load(p, allow_pickle=True)

def event_frames(kind):
    '''Candidate frames (time order) for an event kind.'''
    k = kind.lower()
    coll = {'reward': {'single_reward', 'double_reward', 'money'}, 'banish': {'banish'},
            'timeout': {'timeout'}, 'unbanish': {'unbanish'}}
    if k in coll:
        return np.array(sorted(_ms2f(c['time']) for c in _LOG.get('collected', [])
                               if c.get('effect') in coll[k]))
    if k in ('lick', 'licking'):  return _npz('licking.npz')['bout_spans'][:, 0]
    if k in ('blink',):           return _npz('eye_events.npz')['blink_spans'][:, 0]
    if k in ('saccade', 'sac'):   return _npz('saccades.npz')['saccade']
    if k in ('groom', 'grooming'):
        p = of / 'groom_mask_clean.npy'
        if not p.exists():
            raise FileNotFoundError('groom_mask_clean.npy -- run detect_grooming first')
        m = np.load(p).astype(bool); return np.flatnonzero(m & ~np.r_[False, m[:-1]])
    if k in ('resteer', 'steer', 'joystick'):
        import joymove
        jx, jy = joymove.stick_on_frames(_LOG, _FM); pk, _ = joymove.movement_events(jx, jy, FPS); return pk
    if k in ('motion', 'peak'):
        tot = np.nansum([mag[r] for r in FACIAL], axis=0); return np.argsort(-tot)[:200]
    raise ValueError(f"unknown kind {kind!r}: reward/banish/unbanish/lick/groom/blink/saccade/resteer/motion")

def _rank_roi(kind, rank):
    if rank: return rank
    return 'mouth' if ('lick' in kind.lower() and 'mouth' in mag) else ('paw' if 'paw' in mag else FACIAL[0])
def _val(roi, f, avg_ms):
    if roi not in mag: return np.nan
    if avg_ms <= 0: return float(mag[roi][f])
    h = _span(avg_ms) // 2; a, b = max(LO, f - h), min(HI, f + h + 1); return float(np.nanmean(mag[roi][a:b]))
def _resolve(frames, offset_ms):
    f = np.asarray(frames, int) + _span(offset_ms); return f[(f >= LO) & (f < HI)]
def _offlbl(o): return '' if not o else f' {o:+.0f} ms'
print('helpers ready:  browse(kind, offset_ms=, avg_ms=)  ·  present(frame=|kind=, offset_ms=, avg_ms=)')
print('event kinds  :  reward · banish · unbanish · lick · groom · blink · saccade · resteer · motion')

In [ ]:
def browse(kind='reward', offset_ms=0, avg_ms=0, n=6, rank=None, skip_groom=True, cols=3):
    '''Contact sheet of the top-n candidate frames for an event, ranked by ROI motion. Prints indices.'''
    fr = _resolve(event_frames(kind), offset_ms)
    if len(fr) == 0: raise RuntimeError(f'no {kind} events in the flow window ({WHERE})')
    rr = _rank_roi(kind, rank)
    gm = np.load(of / 'groom_mask_clean.npy').astype(bool) \
        if (skip_groom and (of / 'groom_mask_clean.npy').exists()) else None
    scored = [(_val(rr, int(f), avg_ms), int(f)) for f in fr
              if not (gm is not None and f < len(gm) and gm[f])]
    scored.sort(reverse=True); scored = scored[:n]
    print(f'{kind}: {len(fr)} events | top {len(scored)} by {rr} |flow|'
          + (f', {avg_ms} ms avg' if avg_ms else '') + f' | frames = {[f for _, f in scored]}')
    rows = int(np.ceil(len(scored) / cols)); fig, ax = plt.subplots(rows, cols, figsize=(5 * cols, 3.1 * rows))
    ax = np.atleast_1d(ax).ravel(); cap = cv2.VideoCapture(str(ORIGINAL_VIDEO))
    for a, (sc, f) in zip(ax, scored):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok, im = cap.read()
        if not ok or im is None:                    # frame out of range / wrong video -> skip, don't crash
            a.axis('off'); a.set_title(f'frame {f}: unreadable', fontsize=9); continue
        for nm, r in ROIS.items():
            x1, y1, x2, y2 = r['bbox']; c = tuple(int(z) for z in r['color_bgr'])
            cv2.rectangle(im, (x1, y1), (x2, y2), c, 2)
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.axis('off')
        a.set_title(f'frame {f}  ({rr} {sc:.1f})', fontsize=9)
    for a in ax[len(scored):]: a.axis('off')
    cap.release(); plt.tight_layout(); plt.show(); return [f for _, f in scored]

def _render(best, how, avg_ms, save, outdir=None, save_path=None, fmt=None,
            ref_frame=None, ref_label=None, rank_roi=None):
    cap = cv2.VideoCapture(str(ORIGINAL_VIDEO))
    nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, best - 1); _o0, f0 = cap.read(); _o1, f1 = cap.read(); cap.release()
    if not (_o0 and _o1) or f0 is None or f1 is None:
        raise IOError(f'cannot read frame {best} of {ORIGINAL_VIDEO.name} '
                      f'(video has ~{nfr} frames) -- out of range or wrong video')
    S = 0.5
    g0 = cv2.resize(cv2.cvtColor(f0, cv2.COLOR_BGR2GRAY), None, fx=S, fy=S)
    g1 = cv2.resize(cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY), None, fx=S, fy=S)
    fl = cv2.calcOpticalFlowFarneback(g0, g1, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    fig = plt.figure(figsize=(15, 9)); gs = fig.add_gridspec(2, 2, height_ratios=[3, 1.25], hspace=0.2, wspace=0.16)
    axI = fig.add_subplot(gs[0, :]); axI.imshow(cv2.cvtColor(f1, cv2.COLOR_BGR2RGB)); axI.axis('off')

    ys, xs = np.mgrid[0:fl.shape[0]:14, 0:fl.shape[1]:14]
    u = fl[ys, xs, 0]; v = fl[ys, xs, 1]; spd = np.hypot(u, v)
    xs_full, ys_full = xs / S, ys / S                        # quiver grid in full-res image coords

    # faint GREY background field for context (top movers anywhere in frame)
    bg = spd >= np.percentile(spd, 85)
    axI.quiver(xs_full[bg], ys_full[bg], u[bg], v[bg], color='0.65', angles='xy',
               scale_units='xy', scale=0.12, width=0.0015, alpha=0.35)

    vals = {}
    for roi, r in ROIS.items():
        x1, y1, x2, y2 = r['bbox']; col = [c / 255 for c in r['color_bgr'][::-1]]
        axI.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=col, lw=2.2))
        # arrows whose grid point falls INSIDE this box, kept above the 60th pct of speed WITHIN the
        # box (each box shows its own strongest local vectors), drawn in the box colour
        inside = (xs_full >= x1) & (xs_full <= x2) & (ys_full >= y1) & (ys_full <= y2)
        if inside.any():
            keep = inside & (spd >= np.percentile(spd[inside], 60))
            axI.quiver(xs_full[keep], ys_full[keep], u[keep], v[keep], color=col, angles='xy',
                       scale_units='xy', scale=0.12, width=0.0022, alpha=0.95)
        vals[roi] = _val(roi, best, avg_ms)                   # LABEL = the compute_roi_flow metric
        axI.text(x1, y1 - 6, f'{roi}: {vals[roi]:.2f}', color='white', fontsize=9, fontweight='bold',
                 bbox=dict(facecolor=col, alpha=0.9, pad=1.6, edgecolor='none'))
    axI.set_title(f'{MOUSE_ID}  frame {best}  ({how})\nper-ROI motion (|flow|), arrows coloured by ROI',
                  fontsize=13, fontweight='bold')

    axB = fig.add_subplot(gs[1, 0]); order = sorted(vals, key=lambda kk: np.nan_to_num(vals[kk]))
    axB.barh(range(len(order)), [np.nan_to_num(vals[kk]) for kk in order],
             color=[[c / 255 for c in ROIS[kk]['color_bgr'][::-1]] for kk in order])
    axB.set_yticks(range(len(order))); axB.set_yticklabels(order, fontsize=8)
    axB.set_xlabel('|flow| (avg)' if avg_ms else '|flow| at frame'); axB.grid(alpha=0.2, axis='x')

    axT = fig.add_subplot(gs[1, 1]); a, b = max(LO, best - 90), min(HI, best + 90)
    for roi in FACIAL:
        axT.plot(range(a, b), mag[roi][a:b], lw=1.1,
                 color=[c / 255 for c in ROIS[roi]['color_bgr'][::-1]], label=roi)
    axT.axvline(best, color='k', ls='--', lw=1.2, label='shown frame')

    # REWARD (or whatever event) marker at t=0; the ms top-axis is measured from it when present
    t0 = best
    if ref_frame is not None and a <= ref_frame < b:
        axT.axvline(ref_frame, color='#009E73', ls='-', lw=1.8, label=f'{ref_label or "event"} (t=0)')
        t0 = ref_frame
    # PEAK marker: strongest motion of the ranking ROI (else the facial sum) within the window
    pr = rank_roi if (rank_roi in mag) else None
    seg = mag[pr][a:b] if pr else np.nansum([mag[r][a:b] for r in FACIAL], axis=0)
    pk = a + int(np.nanargmax(seg))
    axT.axvline(pk, color='#D55E00', ls=':', lw=1.8, label=f'peak{" " + pr if pr else ""}')
    dt = (_FM[pk] - _FM[t0]) if a <= t0 < b else np.nan
    if np.isfinite(dt):
        axT.annotate(f'{dt:+.0f} ms', (pk, axT.get_ylim()[1]), fontsize=7, color='#D55E00',
                     ha='center', va='top')
    axT.legend(fontsize=6, ncol=2); axT.grid(alpha=0.2); axT.set_xlabel('frame'); axT.set_ylabel('|flow|')
    # second x-axis in MS relative to t=0 (the reward if given, else the shown frame)
    axT2 = axT.twiny()
    axT2.set_xlim(*[float(_FM[int(np.clip(x, LO, HI - 1))] - _FM[t0]) for x in axT.get_xlim()])
    axT2.set_xlabel(f'ms from {ref_label + " (t=0)" if ref_frame is not None else "shown frame"}',
                    fontsize=8); axT2.tick_params(labelsize=7)

    # SAVE. save_path names the exact file (its suffix = the format, e.g. .pdf/.png/.svg); a directory
    # or fmt= use the auto name roi_motion_frame_<n>.<fmt>. PDF/SVG are VECTOR (crisp for slides/print).
    if save_path or save:
        ext = (fmt or 'png').lstrip('.')
        if save_path:
            p = Path(save_path)
            if not p.suffix: p = p / f'roi_motion_frame_{best}.{ext}'
        else:
            p = (Path(outdir) if outdir else (SESSION_DIR / 'debug')) / f'roi_motion_frame_{best}.{ext}'
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(p, dpi=130, bbox_inches='tight'); print('saved', p)
    plt.show()

def present(frame=None, kind=None, at_ms=None, at_frame=None, offset_ms=0, avg_ms=0, peak_ms=0,
            nth='best', rank=None, save=True, outdir=None, save_path=None, fmt=None):
    '''Full presentation figure for ONE frame. Pick the frame ONE of these ways:
         frame= / at_frame=  -> that exact frame
         at_ms=              -> the frame nearest that log-clock millisecond
         kind=               -> an event kind (reward/lick/resteer/...) chosen by nth + offset_ms
       peak_ms > 0 then SNAPS to the peak-|flow| frame within +/- peak_ms/2 of that centre
       (the best representative frame). avg_ms averages the ROI labels over a centred window.
       On the trace: a black dashed line = the SHOWN frame, a green line = the collection event at
       t=0 (reward/banish/... when kind is one), and an orange dotted * = the motion PEAK; the top
       axis reads ms from t=0 (the event if there is one, else the shown frame).
       SAVING: on by default -> <session>/debug/roi_motion_frame_<frame>.png. Redirect with
         fmt='pdf'                   (auto name, VECTOR pdf; 'svg' also works) in <session>/debug
         outdir='/some/dir'          (auto name there; combine with fmt='pdf')
         save_path='/dir/x.pdf'      (exact file; suffix sets the format) or '/dir' (auto name in it)
         save=False                  (show only, write nothing).'''
    rr = _rank_roi(kind or 'motion', rank)
    ref_frame = ref_label = None
    if frame is not None:      centre, how = int(frame), 'manual'
    elif at_frame is not None: centre, how = int(at_frame), f'frame {int(at_frame)}'
    elif at_ms is not None:    centre, how = _ms2f(at_ms), f'{at_ms:.0f} ms'
    elif kind is not None:
        fr = _resolve(event_frames(kind), offset_ms)
        if len(fr) == 0: raise RuntimeError(f'no {kind} events in the flow window ({WHERE})')
        if nth == 'best':
            centre = int(max(fr, key=lambda f: np.nan_to_num(_val(rr, int(f), avg_ms))))
            how = f'{kind}{_offlbl(offset_ms)} — best by {rr}'
        elif nth in ('first', 'last'):
            centre = int(fr[0] if nth == 'first' else fr[-1]); how = f'{kind}{_offlbl(offset_ms)} — {nth}'
        else:
            centre = int(fr[int(nth)]); how = f'{kind}{_offlbl(offset_ms)} — #{nth}'
        if kind.lower() in ('reward', 'banish', 'unbanish', 'timeout'):   # the collection = t=0
            ev = centre - _span(offset_ms)
            if LO <= ev < HI: ref_frame, ref_label = ev, kind.lower()
    else:
        raise ValueError('give one of: frame=, at_frame=, at_ms=, or kind=')
    best = int(centre)
    if peak_ms:                                              # snap to the local peak = best example frame
        h = _span(peak_ms) // 2; a, b = max(LO, centre - h), min(HI, centre + h + 1)
        seg = mag[rr][a:b] if rr in mag else np.nansum([mag[r][a:b] for r in FACIAL], axis=0)
        best = a + int(np.nanargmax(seg)); how += f' -> peak within {peak_ms:.0f} ms'
    if avg_ms: how += f', {avg_ms:.0f} ms avg'
    _render(best, how, avg_ms, save, outdir, save_path, fmt, ref_frame, ref_label, rr); return best

In [ ]:
# 1) eyeball candidates: the reward APPROACH (500 ms before each reward), ranked by paw motion
_cands = browse('reward', offset_ms=-500)

In [ ]:
# 2) the slide: reward approach, snapped to the local peak, labels averaged over a 1-second window
present(kind='reward', offset_ms=-500, peak_ms=800, avg_ms=1000)

**Ideas — pick the frame by `kind` / `at_ms` / `at_frame`, then shape it with `offset_ms`, `peak_ms`,
`avg_ms`:**

| goal | call |
|---|---|
| the reward approach | `present(kind='reward', offset_ms=-500, avg_ms=1000)` |
| the moment of reward | `present(kind='reward', offset_ms=0)` |
| a licking bout | `present(kind='lick', avg_ms=1000)` |
| active steering | `present(kind='resteer')` |
| **around a specific time (ms)** | `present(at_ms=85000, avg_ms=1000)` |
| **a specific frame** | `present(at_frame=32192, avg_ms=1000)` |
| **best frame near a time** (snap to peak) | `present(at_ms=85000, peak_ms=1000)` |
| reward approach, snapped to peak | `present(kind='reward', offset_ms=-500, peak_ms=800)` |
| the 2nd reward (time order) | `present(kind='reward', nth=1)` |
| compare a few candidates | `browse('reward', offset_ms=-500, n=9)` |

**Saving** (on by default → `<session>/debug/roi_motion_frame_<frame>.png`):

| goal | call |
|---|---|
| **vector PDF** (slides / print) | `present(at_frame=32192, fmt='pdf')` |
| a chosen file (format from the suffix) | `present(at_frame=32192, save_path='/path/fig.pdf')` |
| a chosen folder (auto name) | `present(at_frame=32192, outdir='/path', fmt='pdf')` |
| show only, write nothing | `present(at_frame=32192, save=False)` |

- **`offset_ms`** shifts the centre relative to an event (negative = before).
- **`peak_ms`** then SNAPS to the strongest-motion frame within that window — the best representative
  frame — so you don't have to hand-hunt it.
- **`avg_ms`** averages the ROI *labels* over a centred window (steadier than one noisy frame).
- **On the trace** (bottom-right): the **black dashed** line is the shown frame, the **green** line is
  the collection event at **t=0** (the reward/banish/… when you selected by `kind`), and the **orange
  dotted ★** marks the motion **peak** (labelled with its ms offset from t=0). The **top axis reads ms
  from t=0** — so you can see how far before/after the reward the shown frame and the peak sit.
- The image (arrows) is always the exact frame. Arrows are coloured per ROI and thresholded inside each
  box; the box number is the `compute_roi_flow` metric, not the drawn arrows.

## 8 — Optical-flow overlay VIDEO (for presentation)

A tracked clip: the raw video with each ROI boxed, live **flow arrows** drawn inside it, its mean
`|flow|` printed under the box, and a frame counter — the moving version of section 7's still. Farneback
is recomputed per frame here (self-contained), so it works even before step 4 has been run.

Guarded by **`MAKE_FLOW_VIDEO`** (rendering ~1 min of video is a few minutes of compute). Pick where it
starts (`CLIP_START` — e.g. a frame from `browse()`) and how long (`CLIP_SECONDS`). Writes
`<session>/<mouse>_flow_overlay_<lo>_<hi>.mp4`.

In [ ]:
# the flow-overlay renderer (self-contained; Farneback per frame)
def generate_flow_video(video_path, rois, output_path, n_frames=None, start_frame=0, scale=1.0,
                        arrow_step=15, arrow_threshold=1.0, arrow_scale=3):
    cap = cv2.VideoCapture(str(video_path)); fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * scale); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * scale)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); n_frames = total if n_frames is None else n_frames
    out = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    ok, prev = cap.read()
    if not ok:
        print('could not read the start frame'); cap.release(); out.release(); return None
    prev = cv2.resize(prev, (w, h)); pgray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    t0 = time.time()
    for i in range(n_frames):
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.resize(frame, (w, h)); gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(pgray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        for name, roi in rois.items():
            x1, y1, x2, y2 = [int(v * scale) for v in roi['bbox']]; color = tuple(int(c) for c in roi['color_bgr'])
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, name, (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            for y in range(y1, y2, arrow_step):
                for x in range(x1, x2, arrow_step):
                    if y < flow.shape[0] and x < flow.shape[1]:
                        fx, fy = flow[y, x]
                        if np.hypot(fx, fy) > arrow_threshold:
                            cv2.arrowedLine(frame, (x, y), (int(x + fx * arrow_scale), int(y + fy * arrow_scale)),
                                            color, 1, tipLength=0.3)
            rf = flow[y1:y2, x1:x2]
            cv2.putText(frame, f'{np.mean(np.hypot(rf[..., 0], rf[..., 1])):.2f}', (x1, y2 + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
        cv2.putText(frame, f'Frame: {int(start_frame) + i}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        out.write(frame); pgray = gray
        if i % 500 == 0:
            el = time.time() - t0
            print(f'  {i}/{n_frames}  {el:.0f}s  ETA {el / max(i, 1) * (n_frames - i):.0f}s', flush=True)
    cap.release(); out.release()
    print('done ->', output_path); return output_path
print('generate_flow_video() ready')

In [ ]:
MAKE_FLOW_VIDEO = False                                   # True -> render (a few min for ~1 min of video)
CLIP_START      = PREVIEW_LO                              # first frame (e.g. a frame from browse())
CLIP_SECONDS    = 60                                      # length in seconds (~1 min)
FLOW_VIDEO_SRC  = sess.get('video_noreflection') or ORIGINAL_VIDEO   # glint-free if available

if MAKE_FLOW_VIDEO:
    n = int(round(CLIP_SECONDS * sess['fps']))
    outp = SESSION_DIR / f'{MOUSE_ID}_flow_overlay_{CLIP_START}_{CLIP_START + n}.mp4'
    generate_flow_video(FLOW_VIDEO_SRC, ROIS, outp, n_frames=n, start_frame=CLIP_START)
else:
    print('MAKE_FLOW_VIDEO is False -> not rendering. Set it True (+ CLIP_START / CLIP_SECONDS) '
          'to make a ~1 min tracked clip.')

### 8b — grab ONE frame from the movie as a PNG / PDF

Watching the overlay movie and want a still of a particular frame? **`flow_still(frame, ...)`** renders
exactly that frame in the SAME movie style (ROI boxes + arrows + mean `|flow|` + counter) and saves it,
with the same output options as `present()`:

| goal | call |
|---|---|
| PNG in `<session>/debug/` | `flow_still(5361)` |
| **PDF** (auto name) | `flow_still(5361, fmt='pdf')` |
| a chosen file | `flow_still(5361, save_path='/path/still.pdf')` |
| a chosen folder | `flow_still(5361, outdir='/path', fmt='pdf')` |

*The boxes/arrows/text are drawn into the pixels (cv2), so a `.pdf`/`.svg` just embeds the raster frame.
For a figure whose annotations are true vector line-art, use `present(at_frame=5361, fmt='pdf')` from
section 7 instead — that also gives the bar + ms-axis traces.*

In [ ]:
def flow_still(frame, outdir=None, save_path=None, fmt=None,
               scale=1.0, arrow_step=15, arrow_threshold=1.0, arrow_scale=3, src=None):
    '''Save ONE frame in the flow-overlay MOVIE style (boxes + per-ROI arrows + mean |flow| + counter).
       Output like present(): fmt='pdf' (auto name), save_path='/dir/x.pdf' (exact file, suffix picks
       the format) or '/dir' (auto name in it), else <session>/debug/flow_still_<frame>.png.
       NOTE: the boxes/arrows/text are drawn into the PIXELS, so a .pdf/.svg embeds the raster frame
       (a container, not vector line-art). For vector annotations use present() instead.'''
    src = src or NOREF_VIDEO or ORIGINAL_VIDEO
    cap = cv2.VideoCapture(str(src)); nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame) - 1)
    ok0, f0 = cap.read(); ok1, f1 = cap.read(); cap.release()
    if not (ok0 and ok1) or f0 is None or f1 is None:
        raise IOError(f'cannot read frame {frame} of {Path(src).name} (video has ~{nfr} frames)')
    w = int(f1.shape[1] * scale); h = int(f1.shape[0] * scale); f0 = cv2.resize(f0, (w, h)); f1 = cv2.resize(f1, (w, h))
    flow = cv2.calcOpticalFlowFarneback(cv2.cvtColor(f0, cv2.COLOR_BGR2GRAY),
                                        cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY), None, 0.5, 3, 15, 3, 5, 1.2, 0)
    im = f1.copy()
    for name, roi in ROIS.items():
        x1, y1, x2, y2 = [int(v * scale) for v in roi['bbox']]; color = tuple(int(c) for c in roi['color_bgr'])
        cv2.rectangle(im, (x1, y1), (x2, y2), color, 2)
        cv2.putText(im, name, (x1, max(y1 - 5, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        for y in range(y1, y2, arrow_step):
            for x in range(x1, x2, arrow_step):
                if y < flow.shape[0] and x < flow.shape[1]:
                    fx, fy = flow[y, x]
                    if np.hypot(fx, fy) > arrow_threshold:
                        cv2.arrowedLine(im, (x, y), (int(x + fx * arrow_scale), int(y + fy * arrow_scale)),
                                        color, 1, tipLength=0.3)
        rf = flow[y1:y2, x1:x2]
        cv2.putText(im, f'{np.mean(np.hypot(rf[..., 0], rf[..., 1])):.2f}', (x1, y2 + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
    cv2.putText(im, f'Frame: {int(frame)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # resolve the output path (same rules as present()); .png/.jpg via cv2, .pdf/.svg via matplotlib
    ext = (fmt or 'png').lstrip('.')
    if save_path:
        p = Path(save_path)
        if not p.suffix: p = p / f'flow_still_{int(frame)}.{ext}'
    else:
        p = (Path(outdir) if outdir else (SESSION_DIR / 'debug')) / f'flow_still_{int(frame)}.{ext}'
    p.parent.mkdir(parents=True, exist_ok=True)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', ''):
        cv2.imwrite(str(p), im)
    else:                                   # .pdf / .svg -> embed the raster frame, no whitespace
        figS = plt.figure(figsize=(w / 100, h / 100), dpi=100); axS = figS.add_axes([0, 0, 1, 1])
        axS.imshow(rgb); axS.axis('off'); figS.savefig(str(p), bbox_inches='tight', pad_inches=0)
        plt.close(figS)
    print('saved', p)

    plt.figure(figsize=(12, 6.8)); plt.imshow(rgb); plt.axis('off')
    plt.title(f'{MOUSE_ID}  frame {frame}  (flow-overlay still)'); plt.show()
    return p

# example: a still of frame 5361, saved to <session>/debug/ (pass outdir=/save_path=/fmt='pdf' to redirect)
flow_still(5361)

In [ ]:
# ── eye_still: ONE frame in the ANNOTATED EYE/FACE style (the trial15-saccade look) ──────────────
# Same overlay as the validation CLIP (annotate_clip.py) but for a single still: ROI boxes, the
# BLINK / EYE SQUINT / LICKING / GROOMING / WHISKING pills, the fitted-pupil left_eye inset, and the
# iris-radius SPARKLINE tracked around the chosen frame (white cursor on that frame). It calls
# annotate_clip.render_frame, so this still and the clip draw byte-for-byte the same thing.
import annotate_clip as _ac

_EYE_CTX = None
def eye_still(frame, window_ms=6000, style=None, outdir=None, save_path=None, fmt=None,
              show=True, rebuild=False):
    '''Save ONE frame in the annotated eye/face style, with the iris radius TRACKED over
    +/- window_ms/2 around `frame`. Needs the detector outputs (opticflow/pupil_track.npz +
    eye_events.npz; mouth_state.npz / whisker.npz light the licking/grooming/whisking pills if present).
    Renders from the ORIGINAL glint-intact video (what the fit anchors on).
    Output like flow_still(): fmt='pdf' (auto name, VECTOR container), save_path='/dir/x.pdf' (exact
    file, suffix picks the format) or '/dir' (auto name), else <session>/debug/eye_still_<frame>.png.
    Events: BLINK always; DILATION + CONSTRICTION appear when the session's eye_events.npz carries
    dil/con spans (a stable iris) -- auto-detected, or force style='stable'. On an approximate-iris
    session (e.g. JPAS_0168) dil/con are intentionally not emitted, so only blink shows and
    style='stable' is refused rather than faked.
    style=None auto-detects approx vs stable iris.'''
    global _EYE_CTX
    if _EYE_CTX is None or rebuild or (style is not None and _EYE_CTX['style'] != style):
        try:
            _EYE_CTX = _ac.build_context(str(SESSION_DIR), forced_style=style)
        except SystemExit as ex:                     # resolve_style refuses (e.g. 'stable', no dil/con)
            raise ValueError(str(ex))
        except FileNotFoundError as ex:
            raise FileNotFoundError(f'{ex}\n-> eye_still needs the eye detectors (pupil_track.npz + '
                                    'eye_events.npz). Run the full per-session pipeline first.')
    C = _EYE_CTX; f = int(frame)
    if not (0 <= f < C['n']):
        raise IndexError(f'frame {f} outside the tracked range [0, {C["n"]})')
    half = max(int(round(window_ms / 1000 * FPS)) // 2, 5)     # radius window centred on the frame
    a, b = max(0, f - half), min(C['n'], f + half + 1)
    cap = cv2.VideoCapture(str(ORIGINAL_VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, img = cap.read(); cap.release()
    if not ok or img is None:
        raise IOError(f'cannot read frame {f} of {ORIGINAL_VIDEO.name}')
    _ac.render_frame(img, f, C, a, b)                # the trial15-style overlay for this one frame
    h, w = img.shape[:2]; rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    ext = (fmt or 'png').lstrip('.')                 # same output rules as flow_still()
    if save_path:
        p = Path(save_path)
        if not p.suffix: p = p / f'eye_still_{f}.{ext}'
    else:
        p = (Path(outdir) if outdir else (SESSION_DIR / 'debug')) / f'eye_still_{f}.{ext}'
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', ''):
        cv2.imwrite(str(p), img)
    else:                                            # .pdf / .svg -> embed the raster overlay (container)
        figS = plt.figure(figsize=(w / 100, h / 100), dpi=100); axS = figS.add_axes([0, 0, 1, 1])
        axS.imshow(rgb); axS.axis('off'); figS.savefig(str(p), bbox_inches='tight', pad_inches=0)
        plt.close(figS)
    print('saved', p, f'  [{C["style"]} iris, radius tracked over frames {a}-{b}]')
    if show:
        plt.figure(figsize=(13, 7.3)); plt.imshow(rgb); plt.axis('off')
        plt.title(f'{MOUSE_ID}  frame {f}  (eye/face still, {C["style"]} iris)'); plt.show()
    return p

# example: frame 5361 in the eye/face style, iris radius tracked +/-3 s around it
eye_still(5361)
# a stable-iris animal also lights DILATION / CONSTRICTION pills + shades the sparkline
# (auto-detected when the detector emits dil/con spans, or force it):  eye_still(5361, style='stable')